# 03 · Self-improving loop (RSI) for the complaint tagger
Each round the tagger reads new reviews, keeps the labels it is sure about, asks Qwen about the ones it is unsure about,
retrains, and is **promoted only if it beats the current version** on a fixed test. No human per round; your gold labels
are the fixed anchor, checked at the end.

**Runtime → Change runtime type → T4 GPU.** Run the cells top to bottom. First run: leave `DRY_RUN = True` (about 15 min) to prove
everything works, then set it to `False` and run again from Step R2.

In [ ]:
# STEP R0 · Settings (the only cell you edit)
DRY_RUN = True          # True = tiny test run; False = real run (about 1.5 to 3 hours on a free T4)

ROUNDS      = 1 if DRY_RUN else 5      # max rounds
PER_ROUND   = 100 if DRY_RUN else 1000 # new reviews read per round
EPOCHS      = 1 if DRY_RUN else 8      # same recipe as v1.0 when real
SELF_CAP    = 0.30   # self-labels can be at most 30% of what a round adds
MIN_GAIN    = 0.01   # gate 1: macro F1 must rise by at least this
MAX_DROP    = 0.05   # gate 2: no single label may drop more than this
MAX_CREEP   = 0.20   # gate 3: avg labels per review may not rise more than this above v0 (over-tagging guard)
STOP_AFTER  = 2      # stop after this many rejected rounds in a row

In [ ]:
# STEP R1 · Log in and load every dataset we already have
import re, gc, json, time, numpy as np, pandas as pd, torch
from google.colab import userdata
from huggingface_hub import login, HfApi, hf_hub_download
from datasets import Dataset

login(userdata.get("HF_TOKEN"))                       # token lives in Colab 🔑 Secrets, never in a cell
REPO = "nrr26/yelp-complaint-labels"
api = HfApi()
get = lambda f: hf_hub_download(REPO, f, repo_type="dataset")
LABELS = ["wait_time", "staff_attitude", "food_quality", "order_accuracy",
          "price_value", "hygiene_safety", "ambience", "billing_policy"]
to_set = lambda s: set(re.findall(r"[a-z_]+", str(s))) & set(LABELS)

# 1a. The exact v1.0 train/test split, plus Qwen's labels (gives each review its id)
qwen = pd.read_csv(get("llm_labels_v2.csv"))
by_text = dict(zip(qwen["text"], qwen["id"]))
train = pd.read_parquet(get("data/train-00000-of-00001.parquet"))
test = pd.read_parquet(get("data/test-00000-of-00001.parquet"))
for d in (train, test):
    d["id"] = d["text"].map(by_text).astype("Int64")
    d["y"] = d["label_names"].map(to_set)

# 1b. Every human label we have: old Label Desk + Gold Desk + volunteers (crowd)
votes = {}
def vote(i, labs):
    votes.setdefault(int(i), []).append(set(labs) & set(LABELS))
for r in pd.read_csv(get("human_labels_v1.csv")).fillna("").itertuples():
    vote(r.id, to_set(r.human_labels))
try:
    for r in pd.read_csv(get("gold/gold_desk_labels.csv")).fillna("").itertuples():
        vote(r.id, to_set(r.labels))
except Exception as e:
    print("no Gold Desk file yet:", e)
for f in api.list_repo_files(REPO, repo_type="dataset"):
    if f.startswith("crowd/") and f.endswith(".jsonl"):
        for line in open(get(f)):
            row = json.loads(line)
            vote(row["review_id"], row["labels"])
# several people on one review → keep a label if at least half of them ticked it
human = {i: {l for l in LABELS if 2 * sum(l in v for v in vs) >= len(vs)} for i, vs in votes.items()}

# 1c. Humans win: where a person labelled a training review, use their label instead of Qwen's
has_h = lambda i: pd.notna(i) and int(i) in human
train["y"] = [human[int(i)] if has_h(i) else y for i, y in zip(train["id"], train["y"])]

# 1d. Same 1,800 / 200 train/validation cut as v1.0
sp = Dataset.from_dict({"i": list(range(len(train)))}).train_test_split(test_size=0.1, seed=42)
tr, val = train.iloc[sp["train"]["i"]].reset_index(drop=True), train.iloc[sp["test"]["i"]].reset_index(drop=True)

# 1e. Gate set = test reviews no human touched (Qwen labels). Vault = test reviews a human labelled (sealed till the end)
gold_ids = set(pd.read_csv(get("gold/gold_reviews.csv"))["id"].astype(int))
is_vault = test["id"].map(has_h)
gate = test[~is_vault & ~test["id"].isin(gold_ids)].reset_index(drop=True)
vault = test[is_vault].reset_index(drop=True)
vault["y"] = vault["id"].map(lambda i: human[int(i)])

print(f"human labels: {len(human)} reviews ({sum(map(len, votes.values()))} votes)")
print(f"train {len(tr)} (human-corrected: {tr['id'].map(has_h).sum()}) | val {len(val)} | gate {len(gate)} | vault {len(vault)}")
print("ids missing (should be 0):", train["id"].isna().sum() + test["id"].isna().sum())

In [ ]:
# STEP R2 · Build the pool of NEW reviews the loop will read (never seen in v1.0)
from datasets import load_dataset
ds = load_dataset("Yelp/yelp_review_full", split="train")
neg = ds.filter(lambda x: x["label"] <= 1).filter(lambda x: 200 <= len(x["text"]) <= 1500)   # same rules as v1.0
fresh = neg.shuffle(seed=42).select(range(3500, 3500 + ROUNDS * PER_ROUND * 6)).to_pandas()   # rows after the 3,500 we used

# Cheap restaurant filter: needs 2+ different food/restaurant words (v1.0 used Qwen for this; this saves GPU time)
FOOD = re.compile(r"\b(food|menu|dish|meal|dinner|lunch|breakfast|brunch|waiter|waitress|server|restaurant|appetizer|"
                  r"entree|dessert|chef|kitchen|pizza|burger|sushi|taco|steak|salad|fries|coffee|drink|bartender|table)s?\b", re.I)
fresh = fresh[~fresh["text"].isin(set(qwen["text"]))]
fresh = fresh[fresh["text"].map(lambda t: len({m.lower() for m in FOOD.findall(t)}) >= 2)]
pool = fresh[["text"]].reset_index(drop=True)
print(f"new restaurant-like reviews in the pool: {len(pool):,} (the loop needs up to {ROUNDS * PER_ROUND:,})")

In [ ]:
# STEP R3 · The three tools the loop uses: teacher (Qwen), student (DistilBERT), and the scorer
from transformers import (AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score

# 3a. TEACHER: Qwen with the same prompt v2 that made the v1.0 labels
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEFS = {"wait_time": "slow seating, food or bill; forgotten or ignored by staff",
        "staff_attitude": "rude, dismissive or disrespectful staff (not slowness, not mistakes)",
        "food_quality": "taste, freshness, dryness, cooking, temperature, drinks made badly",
        "order_accuracy": "wrong, missing or sold-out items",
        "price_value": "overpriced, tiny portions, not worth it",
        "hygiene_safety": "dirty, hair or bugs in food, got sick",
        "ambience": "noise, music, seating, temperature, decor",
        "billing_policy": "wrong bill, refunds, reservation rules, manager handling"}
EXAMPLES = [("Our pasta was cold and the server ignored us for 20 minutes.", ["food_quality", "wait_time"]),
            ("Bland, overpriced tacos. Never again.", ["food_quality", "price_value"]),
            ("They forgot my fries and the manager refused to refund me.", ["order_accuracy", "billing_policy"]),
            ("Cute decor and friendly staff, but the music was so loud we couldn't talk.", ["ambience"]),
            ("Meh. Nothing special, just mediocre food.", ["food_quality"])]
RULES_V2 = """- These are 1-2 star reviews, so almost every one has at least one complaint.
  Use [] only if the review truly complains about nothing.
- Vague words count: "sub-par", "mediocre", "not appealing", "awful" about food = food_quality.
- Tag complaints only. Praise ("cute decor", "nice waitress") gets no label.
- Slow service is wait_time, not staff_attitude.
- Wrong, missing or sold-out items are order_accuracy, not food_quality."""

def make_prompt(review):
    label_lines = "\n".join(f"- {k}: {v}" for k, v in DEFS.items())
    shots = "\n".join(f"Review: {t}\nAnswer: {json.dumps(l)}" for t, l in EXAMPLES)
    return (f"Tag this restaurant review with complaint labels.\n\nLabels:\n{label_lines}\n\n"
            f"Rules:\n{RULES_V2}\n\nExamples:\n{shots}\n\nReview: {review}\nAnswer:")

def teacher_label(texts, batch=4):
    """Load Qwen, label the texts, then free the GPU for training."""
    if not texts:
        return []
    qtok = AutoTokenizer.from_pretrained(QWEN)
    qtok.padding_side = "left"
    qlm = AutoModelForCausalLM.from_pretrained(QWEN, dtype=torch.float16, device_map="cuda").eval()
    out = []
    for i in range(0, len(texts), batch):
        chats = [[{"role": "user", "content": make_prompt(t)}] for t in texts[i:i + batch]]
        enc = qtok.apply_chat_template(chats, add_generation_prompt=True, padding=True,
                                       return_tensors="pt", return_dict=True).to("cuda")
        with torch.no_grad():
            g = qlm.generate(**enc, max_new_tokens=60, do_sample=False,
                             pad_token_id=qtok.pad_token_id or qtok.eos_token_id)
        out += qtok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    del qlm
    gc.collect(); torch.cuda.empty_cache()
    return [to_set(o) for o in out]

# 3b. STUDENT: DistilBERT, same recipe as v1.0 (weighted loss, lr 5e-5, max 384 tokens), always trained from scratch
BASE = "distilbert-base-uncased"
btok = AutoTokenizer.from_pretrained(BASE)
onehot = lambda sets: np.array([[1.0 if l in s else 0.0 for l in LABELS] for s in sets], dtype=np.float32)

class Rows(torch.utils.data.Dataset):
    def __init__(self, texts, Y):
        self.enc, self.Y = btok(list(texts), truncation=True, max_length=384), Y
    def __len__(self):
        return len(self.Y)
    def __getitem__(self, i):
        return {**{k: v[i] for k, v in self.enc.items()}, "labels": self.Y[i].tolist()}

def train_student(texts, sets):
    Y = onehot(sets)
    pos = Y.sum(0)
    pw = torch.tensor(np.clip((len(Y) - pos) / np.maximum(pos, 1), 1, 10), dtype=torch.float32)
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE, num_labels=len(LABELS), problem_type="multi_label_classification",
        id2label=dict(enumerate(LABELS)), label2id={n: i for i, n in enumerate(LABELS)})

    class Weighted(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            out = model(**inputs)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(
                out.logits, labels, pos_weight=pw.to(out.logits.device))
            return (loss, out) if return_outputs else loss

    args = TrainingArguments(output_dir="/content/rsi/tmp", num_train_epochs=EPOCHS, learning_rate=5e-5,
                             per_device_train_batch_size=16, fp16=True, save_strategy="no",
                             report_to="none", logging_steps=100, seed=42)
    Weighted(model=model, args=args, train_dataset=Rows(texts, Y),
             data_collator=DataCollatorWithPadding(btok)).train()
    return model.eval()

def predict(model, texts, bs=64):
    probs = []
    for i in range(0, len(texts), bs):
        enc = btok(list(texts[i:i + bs]), truncation=True, max_length=384,
                   padding=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            probs.append(torch.sigmoid(model(**enc).logits.float()).cpu().numpy())
    return np.vstack(probs)

# 3c. SCORER: per-label thresholds tuned on validation, then F1 on the gate set
GRID = np.round(np.arange(0.10, 0.91, 0.05), 2)
def calibrate(P, Y):
    return np.array([GRID[np.argmax([f1_score(Y[:, j], P[:, j] > t, zero_division=0) for t in GRID])]
                     for j in range(len(LABELS))])

def score(P, Y, th):
    pred = (P > th).astype(int)
    return {"macro_f1": f1_score(Y, pred, average="macro", zero_division=0),
            "micro_f1": f1_score(Y, pred, average="micro", zero_division=0),
            "per_label": f1_score(Y, pred, average=None, zero_division=0),
            "avg_labels": pred.sum(1).mean()}

Yval, Ygate = onehot(val["y"]), onehot(gate["y"])
print("✅ teacher, student and scorer ready")

In [ ]:
# STEP R4 · Round 0: train v0 with this exact recipe so every later round is a fair comparison
t0 = time.time()
base_texts, base_sets = tr["text"].tolist(), tr["y"].tolist()
best = train_student(base_texts, base_sets)
best_th = calibrate(predict(best, val["text"].tolist()), Yval)
best_s = v0_s = score(predict(best, gate["text"].tolist()), Ygate, best_th)
v0, v0_th = best, best_th
print(f"v0  macro F1 {best_s['macro_f1']:.3f} | micro F1 {best_s['micro_f1']:.3f} | "
      f"avg labels {best_s['avg_labels']:.2f} | {time.time() - t0:.0f}s")

In [ ]:
# STEP R5 · The loop: read → sort by confidence → teacher labels the unsure → retrain → gate → promote or reject
import os
os.makedirs("/content/rsi", exist_ok=True)
summ = lambda s: {"macro_f1": round(s["macro_f1"], 4), "micro_f1": round(s["micro_f1"], 4), "avg_labels": round(s["avg_labels"], 2)}
log = [{"round": 0, "status": "baseline", **summ(best_s), "train_rows": len(base_texts), "teacher": 0, "self": 0, "why": ""}]
added = pd.DataFrame(columns=["text", "labels", "source", "round"])
cursor, rejects = 0, 0

for k in range(1, ROUNDS + 1):
    t0 = time.time()
    batch = pool.iloc[cursor:cursor + PER_ROUND]
    cursor += PER_ROUND
    if batch.empty:
        print("pool is empty, stopping"); break

    # 1. Read: the current best tagger scores each new review
    P = predict(best, batch["text"].tolist())
    sure = ((P < 0.1) | (P > 0.9)).all(1)                       # confident on all 8 labels
    unsure = (np.abs(P - best_th) < 0.15).any(1) & ~sure          # at least one label near its threshold

    # 2. Teacher labels the unsure ones; the tagger keeps its own labels for the sure ones (capped)
    teach_txt = batch["text"][unsure].tolist()
    teach = pd.DataFrame({"text": teach_txt, "labels": teacher_label(teach_txt), "source": "teacher", "round": k})
    self_rows = pd.DataFrame({"text": batch["text"][sure].values,
                              "labels": [{l for l, p in zip(LABELS, row) if p > 0.9} for row in P[sure]],
                              "source": "self", "round": k})
    cap = int(SELF_CAP / (1 - SELF_CAP) * len(teach))
    self_rows = self_rows.sample(min(len(self_rows), cap), random_state=k)

    # 3. Retrain a candidate from scratch on: original data + everything promoted so far + this round
    cand_added = pd.concat([added, teach, self_rows], ignore_index=True)
    model = train_student(base_texts + cand_added["text"].tolist(), base_sets + cand_added["labels"].tolist())
    th = calibrate(predict(model, val["text"].tolist()), Yval)
    s = score(predict(model, gate["text"].tolist()), Ygate, th)

    # 4. Gate: promote only if all three checks pass
    gain = s["macro_f1"] - best_s["macro_f1"]
    drop = float((best_s["per_label"] - s["per_label"]).max())
    creep = s["avg_labels"] - v0_s["avg_labels"]
    fails = [f"gain {gain:+.3f} < {MIN_GAIN}"] * bool(gain < MIN_GAIN) + \
            [f"a label dropped {drop:.3f}"] * bool(drop > MAX_DROP) + \
            [f"over-tagging +{creep:.2f}"] * bool(creep > MAX_CREEP)
    if not fails:
        del best
        best, best_th, best_s, added, rejects = model, th, s, cand_added, 0
        status = "promoted"
    else:
        del model
        rejects += 1
        status = "rejected"
    gc.collect(); torch.cuda.empty_cache()

    log.append({"round": k, "status": status, **summ(s), "train_rows": len(base_texts) + len(cand_added),
                "teacher": len(teach), "self": len(self_rows), "why": "; ".join(fails)})
    print(f"round {k}: {status.upper()} | macro F1 {s['macro_f1']:.3f} ({gain:+.3f}) | avg labels {s['avg_labels']:.2f} | "
          f"teacher {len(teach)} self {len(self_rows)} skipped {len(batch) - len(teach) - len(self_rows)} | "
          f"{(time.time() - t0) / 60:.1f} min {'| ' + '; '.join(fails) if fails else ''}")

    # 5. Save progress to the private dataset after every round, so nothing is lost if Colab disconnects
    pd.DataFrame(log).to_csv("/content/rsi/rounds.csv", index=False)
    added.assign(labels=added["labels"].map(lambda x: ", ".join(sorted(x)))).to_csv("/content/rsi/added.csv", index=False)
    if not DRY_RUN:
        for f in ("rounds.csv", "added.csv"):
            api.upload_file(path_or_fileobj=f"/content/rsi/{f}", path_in_repo=f"rsi/{f}", repo_id=REPO, repo_type="dataset")

    if rejects >= STOP_AFTER:
        print(f"{STOP_AFTER} rejections in a row, the loop stopped itself"); break

pd.DataFrame(log)

In [ ]:
# STEP R6 · Results: chart per round + open the vault (human labels) once
import matplotlib.pyplot as plt
L = pd.DataFrame(log)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
colors = L["status"].map({"baseline": "#2A7F83", "promoted": "#2A7F83", "rejected": "#C8323C"})
ax[0].plot(L["round"], L["macro_f1"], color="#999", lw=2, zorder=1)
ax[0].scatter(L["round"], L["macro_f1"], c=colors, s=90, zorder=2)
ax[0].set(title="Macro F1 on the gate set (teal = kept, red = rejected)", xlabel="round", ylabel="macro F1")
ax[1].bar(L["round"], L["avg_labels"], color=colors)
ax[1].axhline(1.53, ls="--", color="#E0A22B", label="human average 1.53")
ax[1].set(title="Average labels per review", xlabel="round"); ax[1].legend()
plt.tight_layout(); plt.savefig("/content/rsi/rounds.png", dpi=150); plt.show()

# The vault: reviews people labelled. Small, so read it as a direction, not a final score.
jac = lambda a, b: 1.0 if not a and not b else len(a & b) / len(a | b)
def vault_score(model, th):
    pred = predict(model, vault["text"].tolist()) > th
    sets = [{l for l, on in zip(LABELS, row) if on} for row in pred]
    return np.mean([jac(p, h) for p, h in zip(sets, vault["y"])]), np.mean([len(p) for p in sets])
for name, (m, th) in {"v0": (v0, v0_th), "best": (best, best_th)}.items():
    j, n = vault_score(m, th)
    print(f"{name:>4} vs humans on {len(vault)} vault reviews: Jaccard {j:.2f} | avg labels {n:.2f} (humans {np.mean([len(h) for h in vault['y']]):.2f})")

In [ ]:
# STEP R7 · Keep the best tagger (saved in Colab only; publishing to the model page is a separate, deliberate step)
best.config.label_thresholds = {n: round(float(t), 2) for n, t in zip(LABELS, best_th)}
best.save_pretrained("/content/rsi/best_model"); btok.save_pretrained("/content/rsi/best_model")
print("saved /content/rsi/best_model |", best.config.label_thresholds)